## Sprint 1: Alinhamento e Escopo

In [1]:
# Bibliotecas
import pandas as pd
import numpy as np
import os
from abc import ABC, abstractmethod

import lightgbm as lgb
from sklearn.metrics import mean_absolute_error

In [2]:
# 1. Abstrações (Príncipio do aberto/fechado - OCP)
class DataGenerator(ABC):
    @abstractmethod 
    def generate(self) -> pd.DataFrame:
        pass

class DataLoader(ABC):
    @abstractmethod 
    def load(self, filepath: str) -> pd.DataFrame:
        pass

In [3]:
# 2. Implementações (Responsabilidade única - SRP)
class ProtecDataGenerator(DataGenerator):
    """Classe responsável única por criar a simulação do mercado industrial B2B."""
    def __init__(self, start_date='2023-01-01', end_date='2026-01-01'):
        self.dates = pd.date_range(start=start_date, end=end_date, freq='D')
        np.random.seed(42)

    def generate(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        vendas, compras = [], []
        # SKU 1001: Sensor Ótico (Classe A - Alto Giro) [cite: 77, 91]
        # SKU 2002: Inversor de Frequência (Classe C - Baixo Giro, Alto Custo) [cite: 80, 93]
        produtos = [
            {"id": 1001, "cat": "Sensores", "base": 15, "preco": 120.0, "sazonal": True},
            {"id": 2002, "cat": "Acionamentos", "base": 1, "preco": 3500.0, "sazonal": False}
        ]
        
        for p in produtos:
            for dt in self.dates:
                if p["sazonal"]:
                    fator = 1 + 0.3 * np.sin(2 * dt.dayofyear * np.pi / 365)
                    qtd = int(np.random.poisson(p["base"] * fator))
                else:
                    qtd = 1 if np.random.rand() < 0.15 else 0
                if qtd > 0:
                    vendas.append({"data_registro": dt, "id_produto": p["id"], 
                                   "categoria_produto": p["cat"], "total_vendido": qtd, 
                                   "faturamento_diario": qtd * p["preco"]})
            
            for i in range(30):
                emissao = np.random.choice(self.dates)
                lt = int(np.random.normal(6, 1.2)) if p["id"] == 1001 else int(np.random.normal(30, 5.0))
                compras.append({"id_produto": p["id"], "id_fornecedor": 500 + p["id"], "id_pedido_compra": 10000 + i,
                                "data_emissao_pedido": emissao, "data_recebimento_estoque": emissao + pd.Timedelta(days=max(1, lt)),
                                "lead_time_real": max(1, lt)})
                                
        return pd.DataFrame(vendas), pd.DataFrame(compras)

class LocalCSVLoader(DataLoader):
    """Classe responsável única pelo carregamento e parseamento correto de arquivos CSV."""
    def __init__(self, date_cols: list[str]):
        self.date_cols = date_cols

    def load(self, filepath: str) -> pd.DataFrame:
        if not os.path.exists(filepath):
            raise FileNotFoundError(f"Arquivo não localizado em: {filepath}")
        return pd.read_csv(filepath, parse_dates=self.date_cols)

In [4]:
# 3. Execução do pipeline 

# Criando diretórios e dados
os.makedirs('../data', exist_ok=True)
generator = ProtecDataGenerator()
df_vendas_gen, df_compras_gen = generator.generate()
df_vendas_gen.to_csv('../data/historico_vendas.csv', index=False)
df_compras_gen.to_csv('../data/historico_compras.csv', index=False)

# Carregando dados via loaders especialistas (SOLID)
vendas_loader = LocalCSVLoader(date_cols=['data_registro'])
compras_loader = LocalCSVLoader(date_cols=['data_emissao_pedido', 'data_recebimento_estoque'])

df_vendas = vendas_loader.load('../data/historico_vendas.csv')
df_compras = compras_loader.load('../data/historico_compras.csv')

print(f"[OK] df_vendas carregado com sucesso: {df_vendas.shape[0]} registros.")
print(f"[OK] df_compras carregado com sucesso: {df_compras.shape[0]} registros.")

[OK] df_vendas carregado com sucesso: 1248 registros.
[OK] df_compras carregado com sucesso: 60 registros.


## Sprint 2: Ingestão e Exploração de Dados

In [5]:
#1. Abstração para computação de insights
class DataAnalyzer(ABC):
    @abstractmethod
    def analyze(self, df: pd.DataFrame) -> None:
        pass

#2. Implementações concretas 
class VendasAnalyzer(DataAnalyzer):
    """Classe responsável por extrair a estatística de consumo e volatilidade de demanda"""
    def analyze(self, df: pd.DataFrame) -> None:
        print("EDA de vendas (Métricas de demanda e volatilidade por SKU)")
        print("=" * 60)

    # Agrupamento para calcular métricas diárias essesnciais de demanda
        metrics = df.groupby('id_produto').agg(
            Demanda_media_diaria=('total_vendido', 'mean'),
            Volatilidade_demanda_sigma=('total_vendido', 'std'),
            Faturamento_total=('faturamento_diario', 'sum'),
            Dias_com_vendas=('data_registro', 'count')).reset_index()
    
        # Formatação visual limpa das estatísticas por produto
        for _, row in metrics.iterrows():
            print(f"Produto SKU ID: {int(row['id_produto'])}")
            print(f"Média de Vendas Diárias (D_média): {row['Demanda_media_diaria']:.2f} unidades")
            print(f"Desvio Padrão da Demanda: {row['Volatilidade_demanda_sigma']:.2f}")
            print(f"Faturamento Acumulado no ERP: R$ {row['Faturamento_total']:,.2f}")
            print(f"Presença de Histórico: {int(row['Dias_com_vendas'])} dias registrados")
            print("=" * 60)

class ComprasLeadTimeAnalyzer(DataAnalyzer):
    """Classe responsável única por extrair e auditar a variabilidade logística dos fornecedores"""
    def analyze(self, df: pd.DataFrame) -> None:
        print("EDA de compras (Métricas de Lead Time Real e Confiabilidade)")
        print("=" * 60)

        # Agrupamento para aduitar atrasos e médias reais de recebimento
        metrics = df.groupby('id_produto').agg(
            Lead_time_medio_real=('lead_time_real', 'mean'),
            Volatilidade_fornecedor_sigma=('lead_time_real', 'std'),
            Total_pedidos_auditados=('id_pedido_compra', 'count')).reset_index()
        
        for _, row in metrics.iterrows():
            print(f"Logística para Itens Vinculados ao SKU: {int(row['id_produto'])}")
            print(f"Lead Time Médio Real (LT_médio): {row['Lead_time_medio_real']:.1f} dias")
            print(f"Desvio Padrão Logístico: {row['Volatilidade_fornecedor_sigma']:.2f} dias")
            print(f"Volume de Lotes de Entrada Analisados: {int(row['Total_pedidos_auditados'])} ordens")
            print("=" * 60)
            
#3. Execução do pipeline de exploração
vendas_analise = VendasAnalyzer()
compras_analise = ComprasLeadTimeAnalyzer()

vendas_analise.analyze(df_vendas)
compras_analise.analyze(df_compras)

EDA de vendas (Métricas de demanda e volatilidade por SKU)
Produto SKU ID: 1001
Média de Vendas Diárias (D_média): 14.94 unidades
Desvio Padrão da Demanda: 5.31
Faturamento Acumulado no ERP: R$ 1,966,080.00
Presença de Histórico: 1097 dias registrados
Produto SKU ID: 2002
Média de Vendas Diárias (D_média): 1.00 unidades
Desvio Padrão da Demanda: 0.00
Faturamento Acumulado no ERP: R$ 528,500.00
Presença de Histórico: 151 dias registrados
EDA de compras (Métricas de Lead Time Real e Confiabilidade)
Logística para Itens Vinculados ao SKU: 1001
Lead Time Médio Real (LT_médio): 5.5 dias
Desvio Padrão Logístico: 1.38 dias
Volume de Lotes de Entrada Analisados: 30 ordens
Logística para Itens Vinculados ao SKU: 2002
Lead Time Médio Real (LT_médio): 28.8 dias
Desvio Padrão Logístico: 5.54 dias
Volume de Lotes de Entrada Analisados: 30 ordens


## Sprint 3: Engenharia de Recursos (Features) & Segmentação

In [6]:
#1. Abstração para engenharia de atributos
class FeatureEngineer(ABC):
    @abstractmethod
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        pass

#2. Implementações concretas
class ABCCurveSegmenter(FeatureEngineer):
    """Classe responsável única por aplicar a metolodgia de curva ABC baseada no faturemento"""
    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df_copy = df.copy()

        #1. Calcular o faturamento total por produto
        faturamento_por_sku = df_copy.groupby('id_produto')['faturamento_diario'].sum().reset_index()
        faturamento_por_sku = faturamento_por_sku.sort_values(by='faturamento_diario', ascending=False)

        #2. Calcular a representatividade percentual acumulada
        faturamento_total_geral = faturamento_por_sku['faturamento_diario'].sum()
        faturamento_por_sku['pct_acumulado'] = faturamento_por_sku['faturamento_diario'].cumsum() / faturamento_total_geral

        #3. Definir cortes de mercado (A: até 70%, B: até 90%, C: o restante)
        def determinar_classe(pct):
            if pct <= 0.70: return 'A'
            elif pct <= 0.90: return 'B'
            else: return 'C'

        faturamento_por_sku['classificacao_abc'] = faturamento_por_sku['pct_acumulado'].apply(determinar_classe)

        # Mapear a classificação de volta no DataFrame original
        mapeamento = dict(zip(faturamento_por_sku['id_produto'], faturamento_por_sku['classificacao_abc']))
        df_copy['classificacao_abc'] = df_copy['id_produto'].map(mapeamento)

        return df_copy
    
class TimeSeriesFeatureEngineer(FeatureEngineer):
    """Classe responsável por criar variáveis de histórico (lags, médias móveis e sazonalidade)"""
    def __init__(self, target_col: str = 'total_vendido'):
        self.target_col = target_col

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df_copy = df.copy()

        # Garantir ordenação temporal por produtor para os cálculos de janela estarem corretos
        df_copy = df_copy.sort_values(by=['id_produto', 'data_registro']).reset_index(drop=True)

        #1. Criação de lags (o que vendeu há 1, 2 e 7 dias atrás)
        for lag in [1, 2, 7]:
            df_copy[f'lag_{lag}'] = df_copy.groupby('id_produto')[self.target_col].shift(lag)

        #2. Criação de médias móveis (tendência de consumo dos últimos 3 e 7 dias)
        for window in [3, 7]:
            df_copy[f'media_movel_{window}'] = df_copy.groupby('id_produto')[self.target_col].shift(1).transform(lambda x: x.rolling(window).mean())

        #3. Extração de sazonalidade calendária
        df_copy['dia_da_semana'] = df_copy['data_registro'].dt.dayofweek
        df_copy['mes'] = df_copy['data_registro'].dt.month

        return df_copy


#3. Execução do pipeline
# Passo 1: Aplicar a segmentação ABC
segmenter = ABCCurveSegmenter()
df_com_abc = segmenter.transform(df_vendas)

# Passo 2: Aplciar a Engenharia de recursos temporais
engineer = TimeSeriesFeatureEngineer()
df_final_features = engineer.transform(df_com_abc)

# Exibir amostra dos dados com os novos recursos
print("Sprint 3 - Engenharia de Recursos executada com sucesso!")
print("\n Amostra das novas colunas criadas no DataFrame:")
print(df_final_features[['data_registro', 'id_produto', 'classificacao_abc', 'lag_1', 'media_movel_3', 'dia_da_semana']].tail(3))

Sprint 3 - Engenharia de Recursos executada com sucesso!

 Amostra das novas colunas criadas no DataFrame:
     data_registro  id_produto classificacao_abc  lag_1  media_movel_3  \
1245    2025-12-14        2002                 C    1.0            1.0   
1246    2025-12-17        2002                 C    1.0            1.0   
1247    2025-12-28        2002                 C    1.0            1.0   

      dia_da_semana  
1245              6  
1246              2  
1247              6  


## Sprint 4: Modelagem e Validação

In [7]:
#1. Abstrações
class TimeSeriesValidator(ABC):
    @abstractmethod
    def validate(self, df: pd.DataFrame, model_params: dict) -> dict:
        pass

#2. Implementação concreta (SRP - reponsabilidade única)
class TimeSeriesSplitValidator(TimeSeriesValidator):
    """Classe responsável única por aplicar a validação cruzada temporal sem vazamento de dados."""
    def __init__(self, target_col: str = 'total_vendido', date_col: str = 'data_registro'):
        self.target_col = target_col
        self.date_col = date_col
        # Definindo os recursos preditivos gerados na Sprint 3
        self.features = ['id_produto', 'lag_1', 'lag_2', 'lag_7', 'media_movel_3', 'dia_da_semana', 'mes']

    def validate(self, df: pd.DataFrame, model_params: dict) -> tuple[list, pd.DataFrame]:
        # Remover os primeiros registros que ficaram com valores nulos devido aos Lags
        df_clean = df.dropna().copy()
        
        # Ordenar rigorosamente por data para garantir a divisão temporal
        df_clean = df_clean.sort_values(by=self.date_col).reset_index(drop=True)
        
        # Criar os cortes de tempo (Folds) baseados nos anos do histórico
        # Treino: 2023 e 2024, Validação: 2025 (Simulando teste de estresse temporal)
        limiar_tempo = pd.to_datetime('2025-01-01')
        
        df_train = df_clean[df_clean[self.date_col] < limiar_tempo]
        df_val = df_clean[df_clean[self.date_col] >= limiar_tempo]
        
        X_train, y_train = df_train[self.features], df_train[self.target_col]
        X_val, y_val = df_val[self.features], df_val[self.target_col]
        
        # Treinamento isolado (Etapa 1)
        model = lgb.LGBMRegressor(**model_params)
        model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(stopping_rounds=10, verbose=False)])
        
        # Predição no horizonte futuro de validação
        df_val = df_val.copy()
        df_val['previsao_demanda'] = model.predict(X_val)
        
        # Garantir que previsões não sejam negativas (regra de negócio de estoque)
        df_val['previsao_demanda'] = df_val['previsao_demanda'].clip(lower=0)
        
        return model, df_val
    
#3. Execução do treinamento e validação temporal
# Parâmetros iniciais padrão para o lightGBM (Baseline)
params_baseline = {
    'objective': 'regression',
    'metric': 'mae',
    'learning_rate': 0.05,
    'n_estimators': 100,
    'random_state': 42,
    'verbose': -1
}

# Instanciando o validador temporal especialista
validator = TimeSeriesSplitValidator()
modelo_treinado, df_resultados_val = validator.validate(df_final_features, params_baseline)

# Computando o erro MAE global no período de teste futuro (2025)
erro_global = mean_absolute_error(df_resultados_val['total_vendido'], df_resultados_val['previsao_demanda'])

print("Etapas 1 e 2 Concluídas com Sucesso!")
print(f"Erro Médio Absoluto (MAE) Global no horizonte de 2025: {erro_global:.2f} unidades")

# Quebrando a avaliação por SKU para auditoria de negócio
for sku in df_resultados_val['id_produto'].unique():
    df_sku = df_resultados_val[df_resultados_val['id_produto'] == sku]
    erro_sku = mean_absolute_error(df_sku['total_vendido'], df_sku['previsao_demanda'])
    print(f"MAE para o SKU {int(sku)}: {erro_sku:.2f} unidades")

Etapas 1 e 2 Concluídas com Sucesso!
Erro Médio Absoluto (MAE) Global no horizonte de 2025: 2.95 unidades
MAE para o SKU 1001: 3.37 unidades
MAE para o SKU 2002: 0.20 unidades


### Ajuste de Hiperparâmetros 

In [8]:
#1. Abstração para otimização   
class ModelTuner(ABC):
    @abstractmethod
    def tune(self, df: pd.DataFrame, validator: TimeSeriesValidator) -> dict:
        pass

#2. Implementação concreta
class LightGBMGridTuner(ModelTuner):
    """Classe responsável por iterar hiperparâmetros e encontrar a melhor combinação"""
    def __init__(self):
        # Grade de parâmetros a testar para controlar o ajuste do modelo
        self.param_grid = [
            {'objective': 'regression', 'metric': 'mae', 'learning_rate': 0.01, 'n_estimators': 150, 'verbose': -1, 'random_state': 42},
            {'objective': 'regression', 'metric': 'mae', 'learning_rate': 0.05, 'n_estimators': 100, 'verbose': -1, 'random_state': 42},
            {'objective': 'regression', 'metric': 'mae', 'learning_rate': 0.1, 'n_estimators': 80, 'verbose': -1, 'random_state': 42}
        ]

    def tune(self, df: pd.DataFrame, validator: TimeSeriesValidator) -> dict:
        best_error = float('inf')
        best_params = {}

        print("Iniciando ajuste de hiperparâmetros (Grid Search Temporal)")
        print("=" * 60)

        for i, params in enumerate(self.param_grid):
            # Reutiliza o validator seguro criado na etapa anterior
            _, df_res = validator.validate(df, params)
            current_error = mean_absolute_error(df_res['total_vendido'], df_res['previsao_demanda'])
            
            print(f"Configuração {i+1}, Learning Rate: {params['learning_rate']} -> MAE Global: {current_error:.4f}")

            if current_error < best_error:
                best_error = current_error
                best_params = params

        print("=" * 60)
        print(f"Otimização concluída! Melhor MAE Global encontrado: {best_error:.4f}")
        return best_params

#3. Execução da otimização
tuner = LightGBMGridTuner()
melhores_parametros = tuner.tune(df_final_features, validator)

Iniciando ajuste de hiperparâmetros (Grid Search Temporal)
Configuração 1, Learning Rate: 0.01 -> MAE Global: 3.2150
Configuração 2, Learning Rate: 0.05 -> MAE Global: 2.9536
Configuração 3, Learning Rate: 0.1 -> MAE Global: 2.9547
Otimização concluída! Melhor MAE Global encontrado: 2.9536


## Sprint 5: Integração e Simulação Financeira

In [11]:
#1. Abstração para simulação e exportação
class InventorySimulator(ABC):
    @abstractmethod
    def simulate(self, df_val: pd.DataFrame) -> pd.DataFrame:
        pass

class DataExporter(ABC):
    @abstractmethod
    def export(self, df: pd.DataFrame, filepath: str) -> None:
        pass

#2. Implementações concretas 
class ProtecFinancialSimulator(InventorySimulator):
    """Classe responsável por aplicar as regras de negócio logísticas e calcular o ROI."""
    def __init__(self):
        # Parâmetros de Nível de Serviço (Z) por Curva ABC definidos na Sprint 1
        self.z_scores = {'A': 2.05, 'B': 1.28, 'C': 0.84}
        # Estatísticas logísticas reais calculadas na nossa Sprint 2 (EDA)
        self.logistica_sku = {
            1001: {'lt_medio': 5.5, 'sigma_lt': 1.38, 'custo_unitario': 120.0},
            2002: {'lt_medio': 28.8, 'sigma_lt': 5.54, 'custo_unitario': 3500.0}
        }

    def simulate(self, df_val: pd.DataFrame) -> pd.DataFrame:
        df_sim = df_val.copy()

        # Lista para armazenar as novas colunas calculadas
        estoques_seguranca = []
        custos_seguranca_ia = []
        custos_seguranca_tradicional = []

        for _, row in df_sim.iterrows():
            sku = int(row['id_produto'])
            classe = row['classificacao_abc']

            # Recuperar parâmetros especialistas do produto
            z = self.z_scores.get(classe , 1.0)
            lt_medio = self.logistica_sku[sku]['lt_medio']
            sigma_lt = self.logistica_sku[sku]['sigma_lt']
            custo_peca = self.logistica_sku[sku]['custo_unitario']

            # Volatilidade histórica da demanda (Sprint 2: SKU 1001 = 5.31, SKU 2002 = 0.0)
            sigma_d = 5.31 if sku == 1001 else 0.0
            
            # Demandas Médias: IA (Previsão do LightGBM) vs Tradicional (Média fixa histórica)
            d_media_ia = max(0, row['previsao_demanda'])
            d_media_tradicional = 14.94 if sku == 1001 else 1.00
            
            # 1. Aplicando a Fórmula Estatística de Greasley (Validada na Sprint 1)
            # SS = Z * sqrt( (LT * sigma_D^2) + (D^2 * sigma_LT^2) )
            termo_demanda_ia = lt_medio * (sigma_d ** 2)
            termo_logistica_ia = (d_media_ia ** 2) * (sigma_lt ** 2)
            ss_ia = z * np.sqrt(termo_demanda_ia + termo_logistica_ia)
            
            termo_logistica_trad = (d_media_tradicional ** 2) * (sigma_lt ** 2)
            ss_tradicional = z * np.sqrt(termo_demanda_ia + termo_logistica_trad)
            
            # Regra de negócio calibrada
            if classe == 'C':
                ss_ia = min(ss_ia, 1.0)

            # 2. Traduzindo unidades físicas em Impacto Financeiro (Moeda)
            custo_ss_ia = ss_ia * custo_peca
            custo_ss_trad = ss_tradicional * custo_peca
            
            estoques_seguranca.append(round(ss_ia, 2))
            custos_seguranca_ia.append(round(custo_ss_ia, 2))
            custos_seguranca_tradicional.append(round(custo_ss_trad, 2))

        # Injetando os indicadores na tabela final
        df_sim['estoque_seguranca_ia'] = estoques_seguranca
        df_sim['custo_financeiro_ss_ia'] = custos_seguranca_ia
        df_sim['custo_financeiro_ss_tradicional'] = custos_seguranca_tradicional
        
        # Economia gerada = Custo com método reativo/tradicional - Custo com Inteligência Preditiva
        df_sim['economia_capital_giro'] = df_sim['custo_financeiro_ss_tradicional'] - df_sim['custo_financeiro_ss_ia']
        
        return df_sim
    
class TableauCSVExporter(DataExporter):
    """Classe responsável por fazer o output e persistência dos resultados em disco."""
    def export(self, df: pd.DataFrame, filepath: str) -> None:
        df.to_csv(filepath, index=False)
        print(f"Arquivo unificado exportado para integração com o Tableau: '{filepath}'")

# 3. Execução do pipeline
# Executar a simulação financeira baseada em dados reais de negócio
simulator = ProtecFinancialSimulator()
df_final_tableau = simulator.simulate(df_resultados_val)

# Exportar a base final consolidada para a pasta de visualização do repositório
exporter = TableauCSVExporter()
exporter.export(df_final_tableau, '../src/visualization/dados_dashboard_protec.csv')

# Exibir resumo executivo consolidado da economia do projeto
economia_total = df_final_tableau['economia_capital_giro'].sum()
print("\n" + "="*60)
print("Resumo: Impacto financeiro da solução (ROI)")
print("="*60)
print(f"Economia total estimada em Capital de Giro Liberado: R$ {economia_total:,.2f}")

Arquivo unificado exportado para integração com o Tableau: '../src/visualization/dados_dashboard_protec.csv'

Resumo: Impacto financeiro da solução (ROI)
Economia total estimada em Capital de Giro Liberado: R$ 696,330.91
